In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

In [ ]:
# ── CELL 1 — Imports + Paths ────────────────────────────────────────────────
import numpy as np
import rasterio
from rasterio.merge import merge as rio_merge
from rasterio import windows as rio_windows
import glob, gc, os

BASE       = '/content/drive/MyDrive/AARN_project'
DATA_DIR   = f'{BASE}/dataset/processed/data'   # S2 TIFs + CDL TIFs
OUT_DIR    = f'{BASE}/dataset/processed_new'    # NEW arrays go here
os.makedirs(OUT_DIR, exist_ok=True)

SEED       = 42
CHUNK      = 256   # rows per chunk — RAM safety
TRAIN_PER_CLASS = 240
VAL_PER_CLASS   = 60
TOTAL_PER_CLASS = TRAIN_PER_CLASS + VAL_PER_CLASS  # 300

# Paper Table 2 — exact class counts and CDL codes
REGIONS = {
    'arkansas': {
        's2_glob' : f'{DATA_DIR}/Sentinel2_2021_36steps_Arkansas*.tif',
        'cdl_path': f'{DATA_DIR}/CDL_2021_Labels_Arkansas.tif',
        # CDL_code → label,  class_name,  paper_total
        'classes' : {
            5:  (0, 'Soybeans', 4677),
            3:  (1, 'Rice',     2423),
            1:  (2, 'Corn',     1522),
            2:  (3, 'Cotton',    762),
            -1: (4, 'Others',    616),  # -1 = all other CDL codes
        },
        'n_classes': 5,
    },
    'california': {
        's2_glob' : f'{DATA_DIR}/Sentinel2_2021_36steps_California_Sacramento*.tif',
        'cdl_path': f'{DATA_DIR}/CDL_2021_Labels_California_Sacramento.tif',
        # CDL_code → label, class_name, paper_total
        'classes' : {
            69:  (0, 'Grapes',     2054),
            3:   (1, 'Rice',       2037),
            36:  (2, 'Alfalfa',     974),
            75:  (3, 'Almonds',     783),
            204: (4, 'Pistachios',  640),  # added vs old version
            -1:  (5, 'Others',     3512),  # 6 classes total
        },
        'n_classes': 6,
    },
}

print("✅ Cell 1 done — paths configured")
print(f"   DATA_DIR = {DATA_DIR}")
print(f"   OUT_DIR  = {OUT_DIR}")


In [ ]:
# ── CELL 2 — Mosaic S2 tiles + get global transform ────────────────────────
def get_mosaic_info(s2_glob):
    """Return (merged_transform, total_H, total_W, crs) for the S2 mosaic."""
    tiles = sorted(glob.glob(s2_glob))
    assert len(tiles) > 0, f"No tiles found: {s2_glob}"
    datasets = [rasterio.open(t) for t in tiles]
    _, transform = rio_merge(datasets, method='first')
    # Compute total mosaic dimensions
    from rasterio.merge import merge as _merge
    import rasterio.transform as _rt
    merged_arr, merged_tr = _merge(datasets, method='first')
    H, W = merged_arr.shape[1], merged_arr.shape[2]
    crs = datasets[0].crs
    for ds in datasets:
        ds.close()
    del merged_arr; gc.collect()
    return merged_tr, H, W, crs

print("✅ Cell 2 done — mosaic helper defined")


In [ ]:
# CELL 3 — extract_region() FINAL VERSION
# Fix: S2 extraction uses coordinate sampling, never loads full mosaic into RAM

def extract_region(region, cfg):
    rng = np.random.default_rng(SEED)
    n_classes = cfg['n_classes']
    cdl_codes = cfg['classes']

    # ── Step 1: scan CDL (vectorized) ────────────────────────────────────────
    print(f"\n  Scanning CDL: {cfg['cdl_path']}")
    cdl_src = rasterio.open(cfg['cdl_path'])
    cdl_H, cdl_W = cdl_src.height, cdl_src.width
    print(f"  CDL grid: {cdl_H} × {cdl_W} = {cdl_H*cdl_W/1e6:.1f}M pixels")

    class_pixels = {i: [] for i in range(n_classes)}
    for row_start in range(0, cdl_H, CHUNK):
        row_end = min(row_start + CHUNK, cdl_H)
        win   = rio_windows.Window(0, row_start, cdl_W, row_end - row_start)
        chunk = cdl_src.read(1, window=win)
        remapped = np.full(chunk.shape, -1, dtype=np.int8)
        for code, (lbl, name, _) in cdl_codes.items():
            if code != -1:
                remapped[chunk == code] = lbl
        remapped[(chunk > 0) & (remapped == -1)] = cdl_codes[-1][0]
        rs, cs = np.where(chunk > 0)
        for r, c in zip(rs, cs):
            lbl = int(remapped[r, c])
            if lbl >= 0:
                class_pixels[lbl].append((row_start + r, int(c)))
    cdl_src.close()

    for label_id in range(n_classes):
        name    = [v[1] for k,v in cdl_codes.items() if v[0]==label_id][0]
        paper_n = [v[2] for k,v in cdl_codes.items() if v[0]==label_id][0]
        print(f"  Class {label_id} ({name}): {len(class_pixels[label_id]):,} found  (paper: {paper_n:,})")

   # ── Step 2: sample paper_total per class → 300 train/val + rest → test ──
    # Build lookup: label_id → paper_total (from cfg classes config)
    paper_total = {}
    for code, (lbl, name, total) in cfg['classes'].items():
        paper_total[lbl] = total

    print(f"\n  Sampling paper_total per class (train/val/test from fixed pool)...")
    trainval_locs, test_locs = {}, {}
    for label_id in range(n_classes):
        pixels  = np.array(class_pixels[label_id], dtype=np.int32)
        n_avail = len(pixels)
        n_total = paper_total[label_id]   # e.g. 4677 for Soybeans

        # Sample exactly n_total pixels from CDL (the paper's fixed pool)
        if n_avail < n_total:
            print(f"  ⚠️  Class {label_id}: only {n_avail} available, paper wants {n_total}")
            pool_idx = np.arange(n_avail)
        else:
            pool_idx = rng.choice(n_avail, size=n_total, replace=False)

        pool = pixels[pool_idx]           # the 10,000-pixel pool for this class

        # From pool: first 300 → train/val, rest → test
        trainval_locs[label_id] = pool[:TOTAL_PER_CLASS]   # 300
        test_locs[label_id]     = pool[TOTAL_PER_CLASS:]   # remainder

    del class_pixels; gc.collect()

    # ── Step 3: build train/val index arrays ─────────────────────────────────
    tv_locs_list, tv_y_list, train_indices, val_indices = [], [], [], []
    offset = 0
    for label_id in range(n_classes):
        tv   = trainval_locs[label_id]
        perm = rng.permutation(len(tv))
        tv   = tv[perm]
        for loc in tv[:TRAIN_PER_CLASS]:
            train_indices.append(offset); tv_locs_list.append(loc)
            tv_y_list.append(label_id);   offset += 1
        for loc in tv[TRAIN_PER_CLASS:]:
            val_indices.append(offset);   tv_locs_list.append(loc)
            tv_y_list.append(label_id);   offset += 1

    tv_locs   = np.array(tv_locs_list,  dtype=np.int32)
    tv_y      = np.array(tv_y_list,     dtype=np.int64)
    train_idx = np.array(train_indices, dtype=np.int32)
    val_idx   = np.array(val_indices,   dtype=np.int32)

    test_locs_list, test_y_list = [], []
    for label_id in range(n_classes):
        for loc in test_locs[label_id]:
            test_locs_list.append(loc); test_y_list.append(label_id)
    test_locs_arr = np.array(test_locs_list, dtype=np.int32)
    test_y_arr    = np.array(test_y_list,    dtype=np.int64)

    print(f"\n  Train: {len(train_idx)}  Val: {len(val_idx)}  Test locs: {len(test_locs_arr):,}")

    # ── Step 4: extract S2 spectral — pixel-by-pixel per tile (RAM-safe) ─────
    # Never load full mosaic. Open each tile, read 1×1 windows for pixels in it.
    print(f"\n  Extracting S2 for {len(tv_locs)} train/val pixels (tile-by-tile)...")
    s2_tiles = sorted(glob.glob(cfg['s2_glob']))

    N_tv     = len(tv_locs)
    X_tv     = np.zeros((N_tv, 360), dtype=np.float32)
    filled   = np.zeros(N_tv, dtype=bool)

    # Get CDL→S2 coordinate mapping via first S2 tile transform
    with rasterio.open(s2_tiles[0]) as ref:
        s2_tr  = ref.transform
        s2_crs = ref.crs

    # CDL and S2 are on the same GEE grid → row/col map directly
    # BUT CDL may have different resolution than S2 → need to convert
    # via geographic coordinates to be safe
    with rasterio.open(cfg['cdl_path']) as cdl_ref:
        cdl_tr  = cdl_ref.transform
        cdl_crs = cdl_ref.crs

    # Convert pixel locs from CDL grid → geographic → S2 row/col
    from rasterio.transform import xy as _xy, rowcol as _rowcol
    from pyproj import Transformer as _Transformer

    cdl_xs, cdl_ys = _xy(cdl_tr,
                          tv_locs[:, 0].tolist(),
                          tv_locs[:, 1].tolist())
    cdl_xs = np.array(cdl_xs, dtype=np.float64)
    cdl_ys = np.array(cdl_ys, dtype=np.float64)

    for tile_path in s2_tiles:
        if filled.all():
            break
        with rasterio.open(tile_path) as src:
            tile_tr  = src.transform
            tile_crs = src.crs

            # Reproject coords CDL CRS → tile CRS if needed
            if cdl_crs.to_epsg() != tile_crs.to_epsg():
                t = _Transformer.from_crs(
                    cdl_crs.to_epsg(), tile_crs.to_epsg(), always_xy=True)
                xs_t, ys_t = t.transform(cdl_xs, cdl_ys)
            else:
                xs_t, ys_t = cdl_xs, cdl_ys

            src_rows, src_cols = _rowcol(tile_tr, xs_t, ys_t)
            src_rows = np.array(src_rows)
            src_cols = np.array(src_cols)

            in_bounds = (
                (src_rows >= 0) & (src_rows < src.height) &
                (src_cols >= 0) & (src_cols < src.width) &
                (~filled)
            )

            if not in_bounds.any():
                continue

            idx = np.where(in_bounds)[0]
            print(f"    {tile_path.split('/')[-1]}: {len(idx)} pixels")

            # Read in batches of 200 for speed
            BATCH = 200
            for i in range(0, len(idx), BATCH):
                batch = idx[i:i+BATCH]
                for j in batch:
                    win = rasterio.windows.Window(
                        int(src_cols[j]), int(src_rows[j]), 1, 1)
                    X_tv[j] = src.read(window=win)[:, 0, 0]
                    filled[j] = True

        gc.collect()

    if not filled.all():
        print(f"  ⚠️  {(~filled).sum()} pixels not found in any tile")

    # Scale + reshape
    if X_tv.max() > 10:
        X_tv /= 10000.0
    X_tv   = X_tv.reshape(N_tv, 36, 10)
    mask_tv = np.zeros((N_tv, 36), dtype=np.float32)
    for t in range(36):
        mask_tv[:, t] = (X_tv[:, t, :].max(axis=1) > 0).astype(np.float32)

    missing_rate = 1.0 - mask_tv.mean()
    print(f"  Missing rate (train/val): {missing_rate:.1%}")
    print(f"  X shape: {X_tv.shape}")

    return X_tv, mask_tv, tv_y, tv_locs, train_idx, val_idx, test_locs_arr, test_y_arr


print("✅ Cell 3 FINAL done")

In [ ]:
# DIAGNOSTIC — run before Cell 4
import os

# 1. Check Drive is mounted
print("Drive mounted:", os.path.exists('/content/drive/MyDrive'))

# 2. Check AARN_project exists
print("AARN_project:", os.path.exists('/content/drive/MyDrive/AARN_project'))

# 3. List processed/data/
data_dir = '/content/drive/MyDrive/AARN_project/dataset/processed/data'
if os.path.exists(data_dir):
    files = sorted(os.listdir(data_dir))
    print(f"\nprocessed/data/ ({len(files)} files):")
    for f in files:
        print(f"  {f}")
else:
    print(f"\n❌ Does not exist: {data_dir}")
    # Check what does exist
    base = '/content/drive/MyDrive/AARN_project/dataset'
    if os.path.exists(base):
        print(f"\nContents of dataset/:")
        for f in sorted(os.listdir(base)):
            print(f"  {f}")

In [ ]:
# CELL 4 — Run Arkansas (corrected)
print("\n" + "="*60)
print("ARKANSAS")
print("="*60)

X_ark, mask_ark, y_ark, locs_ark, train_ark, val_ark, test_locs_ark, test_y_ark = \
    extract_region('arkansas', REGIONS['arkansas'])

np.save(f'{OUT_DIR}/new_X_arkansas.npy',                X_ark)
np.save(f'{OUT_DIR}/new_mask_arkansas.npy',             mask_ark)
np.save(f'{OUT_DIR}/new_y_arkansas.npy',                y_ark)
np.save(f'{OUT_DIR}/new_pixel_locations_arkansas.npy',  locs_ark)
np.save(f'{OUT_DIR}/new_train_idx_arkansas.npy',        train_ark)
np.save(f'{OUT_DIR}/new_val_idx_arkansas.npy',          val_ark)
np.save(f'{OUT_DIR}/new_test_locs_arkansas.npy',        test_locs_ark)
np.save(f'{OUT_DIR}/new_test_y_arkansas.npy',           test_y_ark)

print(f"\nArkansas saved ✅")
print(f"  X (train/val): {X_ark.shape}")
print(f"  train: {len(train_ark)}  val: {len(val_ark)}")
print(f"  test locations: {len(test_locs_ark):,}  labels: {len(test_y_ark):,}")
# Verify paper counts
for cls, count in zip(*np.unique(test_y_ark, return_counts=True)):
    names = ['Soybeans','Rice','Corn','Cotton','Others']
    paper = [4377, 2123, 1222, 462, 316]
    print(f"  Class {cls} ({names[cls]}): {count:,}  (paper: {paper[cls]:,})")

del X_ark, mask_ark, locs_ark; gc.collect()

In [ ]:
# CELL 5 — Run California (corrected)
print("\n" + "="*60)
print("CALIFORNIA (Sacramento)")
print("="*60)

X_cal, mask_cal, y_cal, locs_cal, train_cal, val_cal, test_locs_cal, test_y_cal = \
    extract_region('california', REGIONS['california'])

np.save(f'{OUT_DIR}/new_X_california.npy',               X_cal)
np.save(f'{OUT_DIR}/new_mask_california.npy',            mask_cal)
np.save(f'{OUT_DIR}/new_y_california.npy',               y_cal)
np.save(f'{OUT_DIR}/new_pixel_locations_california.npy', locs_cal)
np.save(f'{OUT_DIR}/new_train_idx_california.npy',       train_cal)
np.save(f'{OUT_DIR}/new_val_idx_california.npy',         val_cal)
np.save(f'{OUT_DIR}/new_test_locs_california.npy',       test_locs_cal)
np.save(f'{OUT_DIR}/new_test_y_california.npy',          test_y_cal)

print(f"\nCalifornia saved ✅")
print(f"  X (train/val): {X_cal.shape}")
print(f"  train: {len(train_cal)}  val: {len(val_cal)}")
print(f"  test locations: {len(test_locs_cal):,}")
for cls, count in zip(*np.unique(test_y_cal, return_counts=True)):
    names = ['Grapes','Rice','Alfalfa','Almonds','Pistachios','Others']
    paper = [1754, 1737, 674, 483, 340, 3212]
    print(f"  Class {cls} ({names[cls]}): {count:,}  (paper: {paper[cls]:,})")

del X_cal, mask_cal, locs_cal; gc.collect()

In [ ]:
# CELL 6b — Fix Arkansas normalization only (no re-extraction needed)
import numpy as np

X_ark    = np.load(f'{OUT_DIR}/new_X_arkansas.npy')
mask_ark = np.load(f'{OUT_DIR}/new_mask_arkansas.npy')

print(f"Before: min={X_ark.min():.4f}  max={X_ark.max():.4f}")

# The issue: values like 1.05 → were already divided by 10000 during extraction
# but some raw values were stored as float and slightly above 1.0
# Fix: clip to [0, 1] — physically reflectance cannot exceed 1.0
# Values >1.0 in Sentinel-2 are atmospheric artifacts / calibration noise
X_ark = np.clip(X_ark, 0.0, 1.0)

# Rebuild mask from clipped values
mask_ark = np.zeros((len(X_ark), 36), dtype=np.float32)
for t in range(36):
    mask_ark[:, t] = (X_ark[:, t, :].max(axis=1) > 0).astype(np.float32)

print(f"After:  min={X_ark.min():.4f}  max={X_ark.max():.4f}")
print(f"Missing rate: {1 - mask_ark.mean():.1%}")

np.save(f'{OUT_DIR}/new_X_arkansas.npy',    X_ark)
np.save(f'{OUT_DIR}/new_mask_arkansas.npy', mask_ark)
print("✅ Arkansas re-saved with clipped values")

del X_ark, mask_ark

In [ ]:
# CELL 6c — Fix California normalization
X_cal    = np.load(f'{OUT_DIR}/new_X_california.npy')
mask_cal = np.load(f'{OUT_DIR}/new_mask_california.npy')

print(f"Before: min={X_cal.min():.4f}  max={X_cal.max():.4f}")

X_cal    = np.clip(X_cal, 0.0, 1.0)
mask_cal = np.zeros((len(X_cal), 36), dtype=np.float32)
for t in range(36):
    mask_cal[:, t] = (X_cal[:, t, :].max(axis=1) > 0).astype(np.float32)

print(f"After:  min={X_cal.min():.4f}  max={X_cal.max():.4f}")
print(f"Missing rate: {1 - mask_cal.mean():.1%}")

np.save(f'{OUT_DIR}/new_X_california.npy',    X_cal)
np.save(f'{OUT_DIR}/new_mask_california.npy', mask_cal)
print("✅ California re-saved with clipped values")
del X_cal, mask_cal

In [ ]:
# CELL 6 — Validation (corrected for new file names)
print("\n" + "="*60)
print("VALIDATION")
print("="*60)

checks = [
    ('arkansas',   1200, 300, 5,
     ['Soybeans','Rice','Corn','Cotton','Others'],
     [4377, 2123, 1222, 462, 316]),
    ('california', 1440, 360, 6,
     ['Grapes','Rice','Alfalfa','Almonds','Pistachios','Others'],
     [1754, 1737, 674, 483, 340, 3212]),
]

for region, exp_train, exp_val, exp_classes, names, paper_test in checks:
    X        = np.load(f'{OUT_DIR}/new_X_{region}.npy')
    y        = np.load(f'{OUT_DIR}/new_y_{region}.npy')
    mask     = np.load(f'{OUT_DIR}/new_mask_{region}.npy')
    tr       = np.load(f'{OUT_DIR}/new_train_idx_{region}.npy')
    vl       = np.load(f'{OUT_DIR}/new_val_idx_{region}.npy')
    test_locs = np.load(f'{OUT_DIR}/new_test_locs_{region}.npy')
    test_y    = np.load(f'{OUT_DIR}/new_test_y_{region}.npy')

    errors = []
    if X.shape != (exp_train + exp_val, 36, 10):
        errors.append(f"X shape {X.shape} — expected ({exp_train+exp_val}, 36, 10)")
    if len(tr) != exp_train:
        errors.append(f"train {len(tr)} ≠ {exp_train}")
    if len(vl) != exp_val:
        errors.append(f"val {len(vl)} ≠ {exp_val}")
    if len(np.unique(y)) != exp_classes:
        errors.append(f"{len(np.unique(y))} classes ≠ {exp_classes}")
    if np.isnan(X).any():
        errors.append("X contains NaN")
    if X.max() > 1.0:
        errors.append(f"X not normalized: max={X.max():.2f}")
    if mask.shape != X.shape[:2]:
        errors.append(f"mask shape {mask.shape} ≠ {X.shape[:2]}")
    if test_locs.shape[1] != 2:
        errors.append(f"test_locs shape {test_locs.shape} — expected (N,2)")

    if errors:
        print(f"\n{region.upper()} ❌")
        for e in errors: print(f"  → {e}")
    else:
        print(f"\n{region.upper()} ✅")
        print(f"  X (train/val): {X.shape}  [{X.min():.4f}, {X.max():.4f}]")
        print(f"  mask: {mask.shape}  missing rate: {1-mask.mean():.1%}")
        print(f"  train={len(tr)}  val={len(vl)}")
        print(f"  test locations: {len(test_locs):,}  labels: {len(test_y):,}")

        # Train class distribution
        cls, cnt = np.unique(y[tr], return_counts=True)
        print(f"  Train dist: { {names[c]: int(n) for c,n in zip(cls,cnt)} }")

        # Val class distribution
        cls, cnt = np.unique(y[vl], return_counts=True)
        print(f"  Val dist:   { {names[c]: int(n) for c,n in zip(cls,cnt)} }")

        # Test class distribution vs paper
        print(f"  Test class counts vs paper:")
        for c, pn in enumerate(paper_test):
            actual = (test_y == c).sum()
            diff   = actual - pn
            sign   = '+' if diff >= 0 else ''
            print(f"    {names[c]:<12}: {actual:>8,}  (paper: {pn:>5,}  diff: {sign}{diff:,})")

print(f"\n✅ Phase 1 preprocessing COMPLETE — ready for new_phase1_training.py")

In [ ]:
import os

# List everything currently in processed_new so you can see what's there
IN_DIR = '/content/drive/MyDrive/AARN_project/dataset/processed_new'
files = sorted(os.listdir(IN_DIR))
print(f"{len(files)} files found:")
for f in files:
    size = os.path.getsize(f'{IN_DIR}/{f}') / 1e6
    print(f"  {f}  ({size:.1f} MB)")